[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/05_production_engineering/profiling_optimization.ipynb)

# 07 . Profiling and Optimization

## Concept — Profiling & Performance Optimization (revision notes)

**What it is**
- Profiling = measuring where time and memory actually go in a training/inference step, before changing anything.
- Optimization = removing the measured bottleneck: vectorizing Python loops, avoiding needless copies/syncs, tuning threads and dtypes, or compiling the model with `torch.compile`.

**Why it matters**
- Intuition about performance is usually wrong; the slow part is rarely the layer you suspect (often data loading, a Python loop or a hidden copy).
- Memory, not compute, is often the limit; knowing what saves activations for backward tells you what to checkpoint or shrink.

**When to use**
- Always profile first, with warm-up, on realistic shapes. Optimize the top item in the profile, then re-measure.
- `torch.compile` once the model is stable and runs many iterations (compile cost is paid up front).
- Do not micro-optimize code that takes 1% of the step.

**Math & intuition**
- Amdahl's law: speeding up a part that takes fraction f of runtime by factor s gives total speed-up 1 / ((1 - f) + f / s). Optimizing a 5% part by 10x gains only ~4.7%.
- Time = max(compute, memory traffic) for element-wise ops; fusing ops (what compilers do) saves memory passes, not FLOPs.
- Asynchronous GPU execution: wall-clock timing without `torch.cuda.synchronize()` measures launch time, not run time. On CPU everything is synchronous.

### 1. Timing correctly: warm-up, repeats, and the median

**What this cell does (plain language):** we write a `bench()` helper that discards warm-up runs, repeats the call many times and reports the median and spread. A single `time.time()` around one call is dominated by first-call overheads and noise. (On GPU you would also call `torch.cuda.synchronize()` before reading the clock; we include that guarded call so the helper is GPU-safe.)

In [1]:
import time, os, io, tempfile, warnings, statistics
os.environ.setdefault("KINETO_LOG_LEVEL", "5")   # quiet the profiler backend
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

def bench(fn, n=30, warmup=5):
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    for _ in range(warmup): fn()
    sync(); ts = []
    for _ in range(n):
        t = time.perf_counter(); fn(); sync(); ts.append((time.perf_counter() - t) * 1e3)
    return statistics.median(ts), min(ts), max(ts)

a, b = torch.randn(256, 256), torch.randn(256, 256)
med, lo, hi = bench(lambda: a @ b)
print(f"256x256 matmul: median {med:.3f} ms (min {lo:.3f}, max {hi:.3f})")

256x256 matmul: median 0.074 ms (min 0.068, max 0.130)


### 2. torch.profiler on CPU: where does a training step spend time?

**What this cell does (plain language):** we profile one forward + backward + optimizer step of a small MLP with `torch.profiler.profile`, then print the top operators sorted by total CPU time. `record_function` adds our own labels so the table is organized by phase (forward / backward / optimizer) instead of just raw ATen ops.

In [2]:
from torch.profiler import profile, record_function, ProfilerActivity

model = nn.Sequential(nn.Linear(512, 1024), nn.ReLU(), nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 10))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
x, y = torch.randn(128, 512), torch.randint(0, 10, (128,))

def step():
    with record_function("forward"):
        loss = F.cross_entropy(model(x), y)
    with record_function("backward"):
        opt.zero_grad(); loss.backward()
    with record_function("optimizer"):
        opt.step()

for _ in range(3): step()                                    # warm-up outside the profiler
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    for _ in range(5): step()
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=8, max_name_column_width=38))

--------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
--------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             optimizer         0.29%     168.304us        49.39%      28.521ms       5.704ms             5  
             Optimizer.step#AdamW.step         9.51%       5.489ms        49.10%      28.353ms       5.671ms             5  
                              backward         2.52%       1.456ms        31.10%      17.958ms       3.592ms             5  
autograd::engine::evaluate_function...         0.46%     268.152us        26.43%      15.262ms       1.017ms            15  
                        AddmmBackward0         0.57%     330.277us        24.49%      14.144ms     942.938us            15  


### 3. Reading the profile programmatically

**What this cell does (plain language):** tables are for humans; we can also pull numbers out of `key_averages()` to make decisions. Here we sum time per phase label (our `record_function` names) and report percentages. This is the Amdahl view: which phase deserves attention?

In [3]:
events = {e.key: e for e in prof.key_averages()}
phases = {k: events[k].cpu_time_total for k in ("forward", "backward", "optimizer")}
total = sum(phases.values())
for k, v in phases.items():
    print(f"{k:10s} {v / 1e3:8.2f} ms total over 5 steps  ({100 * v / total:4.1f}%)")
top = sorted((e for e in prof.key_averages() if e.key.startswith("aten::")), key=lambda e: -e.self_cpu_time_total)[:3]
print("top self-time ATen ops:", [(e.key, e.count) for e in top])

forward       11.27 ms total over 5 steps  (19.5%)
backward      17.96 ms total over 5 steps  (31.1%)
optimizer     28.52 ms total over 5 steps  (49.4%)
top self-time ATen ops: [('aten::mm', 25), ('aten::div', 30), ('aten::addmm', 15)]


### 4. Exporting a trace for chrome://tracing / Perfetto

**What this cell does (plain language):** the profiler can write a Chrome-trace JSON that you open in `chrome://tracing` or ui.perfetto.dev to see a timeline. We export to a temp file and report its size (we do not display it). `schedule(wait, warmup, active)` is how you profile only a few steps of a long run without recording everything.

In [4]:
from torch.profiler import schedule
trace_dir = tempfile.mkdtemp()
sched = schedule(wait=1, warmup=1, active=2, repeat=1)       # skip 1 step, warm 1, record 2
with profile(activities=[ProfilerActivity.CPU], schedule=sched) as p2:
    for _ in range(4):
        step(); p2.step()                                    # tell the profiler a step finished
path = os.path.join(trace_dir, "trace.json")
p2.export_chrome_trace(path)
print("trace file written:", os.path.getsize(path) > 0, "| size (KB):", os.path.getsize(path) // 1024)

trace file written: True | size (KB): 177


### 5. Memory tracking: what does autograd keep for backward?

**What this cell does (plain language):** training uses much more memory than inference because autograd saves tensors needed for backward. We use `torch.autograd.graph.saved_tensors_hooks` to count the bytes saved during a forward pass with grad enabled, and compare with `torch.no_grad()` where nothing is saved. We also show the profiler's memory columns.

In [5]:
def saved_bytes(fn):
    total = [0]
    def pack(t):
        total[0] += t.numel() * t.element_size(); return t      # called for every tensor autograd saves
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        fn()
    return total[0]

params = sum(p.numel() * 4 for p in model.parameters())
print(f"parameter bytes          : {params / 1e6:.2f} MB")
print(f"saved for backward (grad): {saved_bytes(lambda: model(x)) / 1e6:.2f} MB")
with torch.no_grad():
    print(f"saved with no_grad       : {saved_bytes(lambda: model(x)) / 1e6:.2f} MB")

with profile(activities=[ProfilerActivity.CPU], profile_memory=True) as pm:
    step()
print(pm.key_averages().table(sort_by="self_cpu_memory_usage", row_limit=4, max_name_column_width=30))

parameter bytes          : 6.34 MB
saved for backward (grad): 6.59 MB
saved with no_grad       : 0.00 MB
------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                          Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                      aten::mm        25.03%       3.002ms        25.05%       3.004ms     600.884us       7.04 MB       7.04 MB             5  
                    aten::sqrt         8.11%     972.314us         8.11%     972.314us     162.052us       6.05 MB       6.05 MB             6  
                     aten::div        11.80%       1.416ms        12.10%       1.452ms     241.936us       6.05 MB       6.05 MB             6  
                   aten::

### 6. Perf bug 1: Python loops over tensor elements

**What this cell does (plain language):** the most common slowdown for newcomers: iterating over rows/elements in Python instead of using one vectorized call. We compute pairwise squared distances between two sets of points with a double Python loop and with broadcasting, verify the results match, and compare times.

In [6]:
P, Q = torch.randn(60, 16), torch.randn(50, 16)

def dist_loop():
    out = torch.empty(60, 50)
    for i in range(60):
        for j in range(50):
            out[i, j] = ((P[i] - Q[j]) ** 2).sum()
    return out

def dist_vec():
    return ((P[:, None, :] - Q[None, :, :]) ** 2).sum(-1)       # broadcast to (60, 50, 16), reduce

assert torch.allclose(dist_loop(), dist_vec(), atol=1e-4)
tl, tv = bench(dist_loop, n=5, warmup=1)[1], bench(dist_vec, n=20)[1]   # [1] = best-of-n, robust to noise
print(f"loop: {tl:.2f} ms | vectorized: {tv:.3f} ms | speed-up x{tl / tv:.0f}")

loop: 29.99 ms | vectorized: 0.049 ms | speed-up x618


### 7. Perf bug 2: growing tensors with torch.cat, and accidental float64

**What this cell does (plain language):** two more classics. (a) Appending to a tensor with `torch.cat` inside a loop copies all previous data each time (quadratic work); collect in a list and cat once, or preallocate. (b) A float64 tensor sneaking in (e.g. from `np.array`) makes every matmul slower and doubles memory.

In [7]:
def cat_in_loop(n=2000):
    out = torch.empty(0, 8)
    for _ in range(n): out = torch.cat([out, torch.ones(1, 8)])   # copies `out` every iteration
    return out
def list_then_cat(n=2000):
    parts = [torch.ones(1, 8) for _ in range(n)]
    return torch.cat(parts)                                         # one copy at the end
assert torch.equal(cat_in_loop(), list_then_cat())
print(f"cat in loop: {bench(cat_in_loop, n=5, warmup=1)[1]:.2f} ms | list then cat: {bench(list_then_cat, n=5, warmup=1)[1]:.2f} ms (best of 5)")

A32, B32 = torch.randn(512, 512), torch.randn(512, 512)
A64, B64 = A32.double(), B32.double()
print(f"matmul fp32: {bench(lambda: A32 @ B32)[1]:.2f} ms | fp64: {bench(lambda: A64 @ B64)[1]:.2f} ms (best of 30)")
print("np.array default dtype float64 -> torch.from_numpy keeps it:", torch.from_numpy(__import__('numpy').zeros(2)).dtype)

cat in loop: 6.18 ms | list then cat: 2.55 ms (best of 5)
matmul fp32: 0.54 ms | fp64: 1.14 ms (best of 30)
np.array default dtype float64 -> torch.from_numpy keeps it: torch.float64


### 8. Perf bug 3: non-contiguous tensors and thread oversubscription

**What this cell does (plain language):** (a) after `transpose`/`permute` a tensor is a non-contiguous view; some ops silently copy it or run slower paths. We time a reduction on a transposed vs contiguous copy. (b) `torch.set_num_threads` controls intra-op parallelism; for small ops more threads can be slower, and running several processes each with all threads oversubscribes the CPU. We sweep thread counts on a medium matmul and restore the default afterwards.

In [8]:
big = torch.randn(2048, 1024)
bt = big.t()                                              # view: shape (1024, 2048), non-contiguous
print("transposed is contiguous:", bt.is_contiguous())
print(f"sum(dim=1) on non-contiguous: {bench(lambda: bt.sum(1))[0]:.3f} ms | on .contiguous(): {bench(lambda: bt.contiguous().sum(1))[0]:.3f} ms (includes the copy)")

default = torch.get_num_threads(); M = torch.randn(1024, 1024)
for n in sorted({1, 2, default}):
    torch.set_num_threads(n)
    print(f"threads={n}: 1024x1024 matmul {bench(lambda: M @ M, n=10)[0]:.2f} ms")
torch.set_num_threads(default)                           # always restore global state
print("default threads restored:", torch.get_num_threads() == default)

transposed is contiguous: False


sum(dim=1) on non-contiguous: 0.185 ms | on .contiguous(): 3.878 ms (includes the copy)


threads=1: 1024x1024 matmul 16.33 ms
threads=2: 1024x1024 matmul 9.05 ms


threads=4: 1024x1024 matmul 16.00 ms
default threads restored: True


### 9. torch.compile with a graceful fallback

**What this cell does (plain language):** `torch.compile` captures the model into a graph and generates fused kernels. It needs a working C++ compiler on CPU and pays a one-off compile cost, so we guard it: if it errors, or no compiler exists, we fall back to eager and say so. We compile a small element-wise function (where fusion helps most), check the outputs match, and report compile time vs steady-state speed.

In [9]:
import shutil
def fused(x):
    return torch.sin(x) ** 2 + torch.cos(x) ** 2 * x + x.tanh()   # many element-wise ops -> fusion candidate

xs = torch.randn(1_000_000)
compiled = fused                                         # default: fall back to eager
try:
    if not (shutil.which("g++") or shutil.which("gcc")):
        raise RuntimeError("no C++ compiler found for the CPU backend")
    cfn = torch.compile(fused)
    t = time.perf_counter(); out = cfn(xs); t_compile = time.perf_counter() - t    # first call = trace + compile
    torch.testing.assert_close(out, fused(xs), atol=1e-4, rtol=1e-4)
    compiled = cfn
    print(f"torch.compile OK | first-call (compile) time: {t_compile:.1f} s")
except Exception as e:
    print("torch.compile unavailable -> using eager fallback:", type(e).__name__, str(e)[:100])

te, tc = bench(lambda: fused(xs), n=10)[0], bench(lambda: compiled(xs), n=10)[0]
print(f"eager: {te:.2f} ms | {'compiled' if compiled is not fused else 'fallback (eager)'}: {tc:.2f} ms")

torch.compile OK | first-call (compile) time: 1.2 s


eager: 64.36 ms | compiled: 8.03 ms


### 10. Break-even: when does compilation pay off?

**What this cell does (plain language):** compile time is a fixed cost, so it only pays off after enough iterations. Using the numbers measured above (if compile succeeded), we compute how many calls are needed before the compiled version has saved more time than the compile cost. If compile fell back to eager, there is nothing to compute and we say so.

In [10]:
if compiled is not fused and te > tc:
    saved_per_call = (te - tc) / 1e3
    print(f"saves {saved_per_call * 1e3:.2f} ms per call -> break-even after ~{t_compile / saved_per_call:,.0f} calls")
elif compiled is not fused:
    print("compiled version was not faster on this machine (memory-bound or small op); break-even never reached here")
else:
    print("running in eager fallback: nothing to compare")
print("rule of thumb: compile long-running training/serving loops, not one-off scripts or tiny ops")

saves 56.34 ms per call -> break-even after ~21 calls
rule of thumb: compile long-running training/serving loops, not one-off scripts or tiny ops


## Common bugs

- **Timing a single call, or including the first call.** Fix: Warm up first, repeat, report median; on GPU use `torch.cuda.synchronize()` around timers.
- **Optimizing without profiling.** Fix: Run `torch.profiler` first; fix the top item; re-measure. Amdahl's law punishes work on a 5% slice.
- **Python `for` loops over tensor rows/elements.** Fix: Vectorize with broadcasting, `torch.where`, `index_select`/`gather`, or batched matmul.
- **`torch.cat` / `list +=` in a loop to grow tensors.** Fix: Append to a Python list and concatenate once, or preallocate and index-assign.
- **Calling `.item()` / `.cpu()` / printing tensors every step on GPU.** Fix: Each forces a device sync; accumulate on-device and log every N steps.
- **Accidental float64 or int64 where float32 is intended.** Fix: Check `.dtype` of data coming from NumPy/pandas and cast early (`.float()`); use `torch.set_default_dtype` deliberately.
- **Recompiling over and over with `torch.compile` (dynamic shapes, Python side effects).** Fix: Keep shapes stable, mark dynamic dims, avoid data-dependent Python control flow; check recompiles via `TORCH_LOGS=recompiles`.
- **Using too many threads/workers per process.** Fix: Set `torch.set_num_threads` and `DataLoader(num_workers=...)` so total threads <= cores.
- **Forgetting `torch.no_grad()` / `inference_mode` at eval time.** Fix: Autograd saves activations when grad is enabled; wrap evaluation to save memory and time.
- **Profiling with the profiler left on during the real run.** Fix: Use `schedule(wait, warmup, active)` to profile a few steps only; profiling adds overhead.

## Exercises

**Exercise 1.** Use `bench` to compare `torch.relu(x)` against `x.clamp(min=0)` on a 1M-element tensor. Which is faster, and are results equal?

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

Both are one fused memory pass; any difference is small and machine dependent.

In [12]:
# Solution 1
xr = torch.randn(1_000_000)
print(torch.equal(torch.relu(xr), xr.clamp(min=0)))
print(bench(lambda: torch.relu(xr))[0], bench(lambda: xr.clamp(min=0))[0])

True


7.998708499940221 7.996363999836831


**Exercise 2.** Profile only the forward pass of `model` (inference with `no_grad`) and print the three most expensive operators by `self_cpu_time_total`.

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
with profile(activities=[ProfilerActivity.CPU]) as pf, torch.no_grad():
    for _ in range(5): model(x)
print(pf.key_averages().table(sort_by="self_cpu_time_total", row_limit=3))

----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
           aten::addmm        47.31%     142.790ms        74.15%     223.796ms      14.920ms            15  
           aten::copy_        26.80%      80.896ms        26.80%      80.896ms       5.393ms            15  
       aten::clamp_min        25.59%      77.232ms        25.59%      77.232ms       7.723ms            10  
----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
Self CPU time total: 301.825ms



**Exercise 3.** Compute Amdahl's law: if the optimizer step is 20% of runtime and you make it 4x faster, what total speed-up do you get?

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

Only ~1.18x overall: a big speed-up of a small fraction gives a small total win.

In [16]:
# Solution 3
f, s = 0.2, 4
print(1 / ((1 - f) + f / s))   # 1.176

1.176470588235294


**Exercise 4.** Rewrite a loop that computes `[row.sum() for row in M]` as one vectorized call and verify equality and speed.

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

In [18]:
# Solution 4
M = torch.randn(2000, 100)
slow = torch.stack([r.sum() for r in M]); fast = M.sum(1)
print(torch.allclose(slow, fast, atol=1e-5), bench(lambda: torch.stack([r.sum() for r in M]), n=5)[0], bench(lambda: M.sum(1))[0])

True 5.188747999909538 7.999526000048718


**Exercise 5.** Measure peak 'saved for backward' memory with batch sizes 32, 64, 128 using `saved_bytes`. Is the total proportional to batch size? (Look at the increments.)

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

Saved memory = a constant part (the weights that linear layers save for backward) + a part linear in batch size (activations): doubling the batch adds a growing increment, the weights do not.

In [20]:
# Solution 5
for bs in (32, 64, 128):
    xb = torch.randn(bs, 512)
    print(bs, saved_bytes(lambda: model(xb)) / 1e6, "MB")

32 4.825088 MB
64 5.414912 MB
128 6.59456 MB


**Exercise 6.** Use `torch.utils.benchmark.Timer` (which handles warm-up and thread settings) to time `a @ b` and print its `median`.

In [21]:
# Your attempt for Exercise 6 here

**Solution 6**

In [22]:
# Solution 6
from torch.utils.benchmark import Timer
t = Timer(stmt="a @ b", globals={"a": a, "b": b})
print(t.blocked_autorange(min_run_time=0.2).median * 1e3, "ms")

0.3025147400012429 ms
